# Chapter 1: What is Machine Learning?

## Programming vs Machine Learning

**Traditional programming**: Human writes rules → Computer follows rules
```
if email contains "free money" → spam
if email contains "meeting" → not spam
```

**Machine learning**: Human gives examples → Computer discovers rules
```
1000 spam emails + 1000 good emails → ML finds patterns itself
```

The key difference: **we don't tell the computer HOW to decide. We show it examples, and it figures out the pattern.**

---
## A Simple Example: Is this fruit an apple or orange?

Traditional approach: write rules by hand.

ML approach: give the computer measurements of apples and oranges, let it find the boundary.

In [ ]:
# Traditional programming: WE write the rules
def classify_fruit_traditional(weight, texture):
    """texture: 0=smooth, 1=bumpy"""
    if weight > 160 and texture == 1:
        return "orange"
    else:
        return "apple"

# These rules work for simple cases...
print("Traditional (hand-written rules):")
print(f"  weight=150, smooth → {classify_fruit_traditional(150, 0)}")
print(f"  weight=170, bumpy  → {classify_fruit_traditional(170, 1)}")
print(f"  weight=140, bumpy  → {classify_fruit_traditional(140, 1)}")
print()
print("Problem: What about a 155g bumpy fruit?")
print("We'd have to keep adding more and more rules...")

In [ ]:
# Machine learning: the COMPUTER finds the rules
from sklearn.tree import DecisionTreeClassifier

# Training data: [weight, texture] → label
# texture: 0=smooth, 1=bumpy
features = [
    [140, 0],  # apple
    [130, 0],  # apple
    [150, 0],  # apple
    [170, 1],  # orange
    [180, 1],  # orange
    [160, 1],  # orange
]
labels = ["apple", "apple", "apple", "orange", "orange", "orange"]

# Train: let the computer find patterns
model = DecisionTreeClassifier()
model.fit(features, labels)

# Predict: the computer uses patterns it found
print("ML model (learned rules):")
test_cases = [[150, 0], [170, 1], [140, 1], [155, 1]]
for tc in test_cases:
    pred = model.predict([tc])[0]
    texture = "smooth" if tc[1] == 0 else "bumpy"
    print(f"  weight={tc[0]}, {texture:6s} → {pred}")

print("\nThe computer figured out the rules itself!")
print("We never told it 'bumpy means orange' — it discovered that.")

---
## The 3 Types of Machine Learning

| Type | What you give | What it learns | Example |
|------|--------------|----------------|----------|
| **Supervised** | Input + correct answer | Input → answer mapping | Spam detection |
| **Unsupervised** | Input only (no answers) | Hidden patterns/groups | Customer segmentation |
| **Reinforcement** | Environment + rewards | Strategy to maximize reward | Game playing AI |

Most of this book focuses on **supervised learning** — it's the most common and practical.

In [ ]:
# Supervised learning example: we GIVE the answers during training
print("=" * 50)
print("SUPERVISED LEARNING")
print("=" * 50)
print()
print("Training data (input → correct answer):")
print("  [140g, smooth] → apple    ← we tell it this")
print("  [170g, bumpy]  → orange   ← we tell it this")
print("  [130g, smooth] → apple    ← we tell it this")
print()
print("After training:")
print("  [155g, bumpy]  → ???      ← model predicts on its own")
print()
print("Two sub-types:")
print("  Regression:      predict a NUMBER    (house price: $350,000)")
print("  Classification:  predict a CATEGORY  (spam / not spam)")

In [ ]:
import numpy as np

# Unsupervised learning example: NO answers given
print("=" * 50)
print("UNSUPERVISED LEARNING")
print("=" * 50)
print()
print("Training data (input ONLY, no labels):")
print("  Customer A: age=25, spending=$200/month")
print("  Customer B: age=65, spending=$50/month")
print("  Customer C: age=23, spending=$180/month")
print("  Customer D: age=60, spending=$45/month")
print()

from sklearn.cluster import KMeans

customers = np.array([[25, 200], [65, 50], [23, 180], [60, 45], [28, 220], [62, 55]])
model = KMeans(n_clusters=2, random_state=0, n_init=10)
model.fit(customers)

print("ML discovers 2 groups on its own:")
for i, (cust, label) in enumerate(zip(customers, model.labels_)):
    print(f"  Customer {i+1}: age={cust[0]}, spending=${cust[1]:>3} → Group {label}")
print()
print("Nobody told it what the groups mean — it found")
print("'young high-spenders' vs 'older low-spenders' by itself.")

---
## The ML Workflow

Every ML project follows roughly the same steps:

```
1. Collect data        → gather examples
2. Prepare data        → clean, normalize, split into train/test
3. Choose a model      → decision tree? neural network? regression?
4. Train               → feed training data, model learns patterns
5. Evaluate            → test on data the model hasn't seen
6. Tune                → adjust and repeat until good enough
```

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Full ML workflow in one cell
print("=" * 50)
print("COMPLETE ML WORKFLOW")
print("=" * 50)

# 1. Collect data
print("\n1. COLLECT DATA")
from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris.data, iris.target
print(f"   {len(X)} flower measurements, {len(iris.feature_names)} features each")
print(f"   Features: {iris.feature_names}")
print(f"   Classes: {list(iris.target_names)}")

# 2. Prepare data (split into train/test)
print("\n2. PREPARE DATA")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set:     {len(X_test)} samples (held back for evaluation)")

# 3. Choose a model
print("\n3. CHOOSE MODEL")
model = DecisionTreeClassifier(random_state=42)
print(f"   Using: Decision Tree")

# 4. Train
print("\n4. TRAIN")
model.fit(X_train, y_train)
print(f"   Model learned from {len(X_train)} examples")

# 5. Evaluate
print("\n5. EVALUATE")
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"   Accuracy on test data: {accuracy:.1%}")
print(f"   ({int(accuracy * len(y_test))}/{len(y_test)} correct)")

# Show some predictions
print("\n   Sample predictions:")
for i in range(5):
    actual = iris.target_names[y_test[i]]
    predicted = iris.target_names[predictions[i]]
    match = "✓" if actual == predicted else "✗"
    print(f"   {match} Predicted: {predicted:<12} Actual: {actual}")

---
## Train vs Test: Why We Split Data

Imagine a student who memorizes every answer in the textbook but can't solve new problems. That's **overfitting**.

We split data into:
- **Training set** (70-80%): the model learns from this
- **Test set** (20-30%): we check if the model can handle NEW data

A good model performs well on **both** training AND test data.

In [ ]:
# Overfitting demonstration
print("=" * 50)
print("OVERFITTING vs GOOD FIT")
print("=" * 50)

# Overfit model: memorizes training data (no limits on complexity)
overfit_model = DecisionTreeClassifier(random_state=42)  # no depth limit
overfit_model.fit(X_train, y_train)
train_acc = accuracy_score(y_train, overfit_model.predict(X_train))
test_acc = accuracy_score(y_test, overfit_model.predict(X_test))
print(f"\n  Unrestricted tree:")
print(f"    Training accuracy: {train_acc:.1%}  (memorized!)")
print(f"    Test accuracy:     {test_acc:.1%}")

# Simpler model: generalizes better
simple_model = DecisionTreeClassifier(max_depth=2, random_state=42)
simple_model.fit(X_train, y_train)
train_acc2 = accuracy_score(y_train, simple_model.predict(X_train))
test_acc2 = accuracy_score(y_test, simple_model.predict(X_test))
print(f"\n  Simple tree (max_depth=2):")
print(f"    Training accuracy: {train_acc2:.1%}")
print(f"    Test accuracy:     {test_acc2:.1%}")

print(f"\n  Key insight:")
print(f"  100% on training doesn't mean the model is good.")
print(f"  What matters is performance on UNSEEN data (test set).")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **ML vs Programming** | We give examples, not rules |
| **Supervised** | Input + correct answer → learns mapping |
| **Unsupervised** | Input only → discovers hidden patterns |
| **Regression** | Predict a number |
| **Classification** | Predict a category |
| **Train/Test split** | Always evaluate on unseen data |
| **Overfitting** | Memorizing ≠ learning |

**Next chapter**: Regression — predicting numbers (linear, polynomial)